# LLM-Based Root Cause Analysis for 5G Networks

**Qwen2.5-7B-Instruct | QLoRA Fine-Tuning | Eight-Class Diagnosis**

This experiment investigates whether parameter-efficient fine-tuning enables an instruction-tuned language model to identify predefined causes of throughput degradation from structured 5G radio-network diagnostics.

**Analysis pipeline:** 69 diagnostic features → 18 previously selected features → structured diagnostic prompt → Qwen2.5-7B-Instruct → QLoRA fine-tuning → root-cause prediction (C1–C8).

The notebook includes the original data splits, base-model evaluation, supervised fine-tuning, validation and held-out test evaluation, and saved prediction metrics. The recorded outputs are retained.


In [ ]:
#%pip install -q -U transformers datasets peft accelerate bitsandbytes scikit-learn pandas

## 1. Imports and experiment configuration


In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
import inspect
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    recall_score,
)
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

RANDOM_SEED = 42
MODEL_NAME = "/home/azaddel/models/Qwen2.5-7B-Instruct"
LABELS = [f"C{i}" for i in range(1, 9)]

DATA_DIR = Path(".")

USE_4BIT = True
MAX_LENGTH = 1056

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

print("Model:", MODEL_NAME)
print("Fine-tuning mode:", "QLoRA (4-bit base + LoRA adapters)" if USE_4BIT else "Standard LoRA")


/home/azaddel/conda-envs/qwen5g/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
GPU: NVIDIA GeForce RTX 3080 Ti
GPU memory: 11.63 GB
Model: /home/azaddel/models/Qwen2.5-7B-Instruct
Fine-tuning mode: QLoRA (4-bit base + LoRA adapters)


## 2. Dataset loading and Train / Validation / Test splits

The experiment uses the existing incident-level split assignments throughout model development and evaluation.


In [2]:
FEATURE_FILE = DATA_DIR / "all_2400_incidents_69_features.csv"
SPLIT_FILE = DATA_DIR / "split_assignments.csv"
TARGET_FILE = DATA_DIR / "train.csv"

for p in [FEATURE_FILE, SPLIT_FILE, TARGET_FILE]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing file: {p}\n"
            "Place the required CSV files in DATA_DIR or update DATA_DIR."
        )

features_df = pd.read_csv(FEATURE_FILE)
splits_df = pd.read_csv(SPLIT_FILE)

targets_df = (
    pd.read_csv(TARGET_FILE, usecols=["ID", "answer"])
    .rename(columns={"ID": "incident_id", "answer": "gold_label"})
)

data = (
    features_df
    .merge(splits_df, on="incident_id", validate="one_to_one")
    .merge(targets_df, on="incident_id", validate="one_to_one")
)

assert len(data) == 2400
assert set(data["gold_label"].unique()) == set(LABELS)

train_df = data[data["split"] == "train"].reset_index(drop=True)
val_df = data[data["split"] == "val"].reset_index(drop=True)
test_df = data[data["split"] == "test"].reset_index(drop=True)

print("All incidents:", len(data))
print("Train         :", len(train_df))
print("Validation    :", len(val_df))
print("Test          :", len(test_df))

print("\nOverall label counts:")
display(data["gold_label"].value_counts().sort_index().to_frame("count"))


All incidents: 2400
Train         : 1727
Validation    : 339
Test          : 334

Overall label counts:


,count
gold_label,
C1,264
C2,320
C3,330
C4,283
C5,352
C6,225
C7,349
C8,277


## 3. Selected diagnostic features

The input representation consists of 18 diagnostic features identified in a preceding feature-selection study, drawn from the original 69 features.


In [3]:
SELECTED_FEATURES = [
    "sig__affected_min_rsrp_dbm",
    "tilt__affected_mean_vertical_abs_offset_deg",
    "sig__mean_rsrp_dbm",
    "tilt__max_serving_total_downtilt_deg",
    "sig__affected_mean_rsrp_dbm",
    "mob__serving_pci_change_rate_per_second",
    "trn__mean_throughput_delta_when_target_was_prev_neighbor_mbps",
    "trn__max_throughput_delta_when_target_was_prev_neighbor_mbps",
    "ovl__affected_mean_best_noncolocated_neighbor_delta_vs_serving_db",
    "trn__target_was_top1_previous_neighbor_count",
    "ovl__max_noncolocated_neighbor_delta_vs_serving_db",
    "ovl__affected_max_noncolocated_neighbor_delta_vs_serving_db",
    "trn__max_target_prev_neighbor_delta_vs_old_serving_db",
    "ovl__noncolocated_matched_neighbor_slot_count",
    "tilt__mean_serving_total_downtilt_deg",
    "tilt__affected_mean_azimuth_abs_offset_deg",
    "mob__affected_mean_speed_kmh",
    "res__mean_rb",
]

missing = [f for f in SELECTED_FEATURES if f not in data.columns]
if missing:
    raise ValueError(f"Selected features missing from dataset: {missing}")

print("Selected features:", len(SELECTED_FEATURES))
for i, f in enumerate(SELECTED_FEATURES, 1):
    print(f"{i:2d}. {f}")


Selected features: 18
 1. sig__affected_min_rsrp_dbm
 2. tilt__affected_mean_vertical_abs_offset_deg
 3. sig__mean_rsrp_dbm
 4. tilt__max_serving_total_downtilt_deg
 5. sig__affected_mean_rsrp_dbm
 6. mob__serving_pci_change_rate_per_second
 7. trn__mean_throughput_delta_when_target_was_prev_neighbor_mbps
 8. trn__max_throughput_delta_when_target_was_prev_neighbor_mbps
 9. ovl__affected_mean_best_noncolocated_neighbor_delta_vs_serving_db
10. trn__target_was_top1_previous_neighbor_count
11. ovl__max_noncolocated_neighbor_delta_vs_serving_db
12. ovl__affected_max_noncolocated_neighbor_delta_vs_serving_db
13. trn__max_target_prev_neighbor_delta_vs_old_serving_db
14. ovl__noncolocated_matched_neighbor_slot_count
15. tilt__mean_serving_total_downtilt_deg
16. tilt__affected_mean_azimuth_abs_offset_deg
17. mob__affected_mean_speed_kmh
18. res__mean_rb


## 4. Root-cause definitions and structured diagnostic prompts


In [4]:
ROOT_CAUSE_DESCRIPTIONS = {
    "C1": "The serving cell's downtilt angle is too large, causing weak coverage at the far end.",
    "C2": "The serving cell's coverage distance exceeds 1 km, resulting in over-shooting.",
    "C3": "A neighboring cell provides higher throughput.",
    "C4": "Non-colocated co-frequency neighboring cells cause severe overlapping coverage.",
    "C5": "Frequent handovers degrade performance.",
    "C6": "A neighbor cell and the serving cell have the same PCI mod 30, leading to interference.",
    "C7": "Test vehicle speed exceeds 40 km/h, impacting user throughput.",
    "C8": "Average scheduled RBs are below 160, affecting throughput.",
}

root_cause_text = "\n".join(
    f"{label}: {description}"
    for label, description in ROOT_CAUSE_DESCRIPTIONS.items()
)

SYSTEM_PROMPT = f"""You are an expert 5G radio-network root cause analysis assistant.

A network incident has experienced throughput degradation.
You will receive structured diagnostic features extracted from network measurements.

Identify the single most likely root cause.

Possible root causes:
{root_cause_text}

Return ONLY one label: C1, C2, C3, C4, C5, C6, C7, or C8.
Do not provide an explanation."""

In [5]:
FEATURE_DESCRIPTIONS = {

    "sig__affected_min_rsrp_dbm":
        "Minimum serving-cell RSRP during throughput-affected observations (dBm).",

    "tilt__affected_mean_vertical_abs_offset_deg":
        "Mean absolute vertical angular offset between the UE direction and the serving-cell beam direction during throughput-affected observations (deg).",

    "sig__mean_rsrp_dbm":
        "Mean serving-cell RSRP across all observations (dBm).",

    "tilt__max_serving_total_downtilt_deg":
        "Maximum serving-cell total downtilt, computed as mechanical tilt plus interpreted digital tilt (deg).",

    "sig__affected_mean_rsrp_dbm":
        "Mean serving-cell RSRP during throughput-affected observations (dBm).",

    "mob__serving_pci_change_rate_per_second":
        "Number of serving PCI changes divided by the incident duration in seconds (changes/s).",

    "trn__mean_throughput_delta_when_target_was_prev_neighbor_mbps":
        "Mean throughput change across serving-PCI transitions where the target PCI was observed as a neighbor in the immediately preceding observation (Mbps).",

    "trn__max_throughput_delta_when_target_was_prev_neighbor_mbps":
        "Maximum throughput change across serving-PCI transitions where the target PCI was observed as a neighbor in the immediately preceding observation (Mbps).",

    "ovl__affected_mean_best_noncolocated_neighbor_delta_vs_serving_db":
        "Mean BRSRP-minus-serving-RSRP difference of the strongest matched non-colocated neighbor during throughput-affected observations (dB).",

    "trn__target_was_top1_previous_neighbor_count":
        "Count of serving-PCI transitions where the target PCI was the strongest (Top-1) neighbor by BRSRP in the immediately preceding observation.",

    "ovl__max_noncolocated_neighbor_delta_vs_serving_db":
        "Maximum BRSRP-minus-serving-RSRP difference among matched non-colocated neighbors across all observations (dB).",

    "ovl__affected_max_noncolocated_neighbor_delta_vs_serving_db":
        "Maximum BRSRP-minus-serving-RSRP difference among matched non-colocated neighbors during throughput-affected observations (dB).",

    "trn__max_target_prev_neighbor_delta_vs_old_serving_db":
        "Maximum BRSRP-minus-old-serving-RSRP difference for transitions where the target PCI was observed as a neighbor immediately before the transition (dB).",

    "ovl__noncolocated_matched_neighbor_slot_count":
        "Number of neighbor slots matched to Engineering cells and identified as non-colocated with the serving cell.",

    "tilt__mean_serving_total_downtilt_deg":
        "Mean serving-cell total downtilt, computed as mechanical tilt plus interpreted digital tilt, across all observations (deg).",

    "tilt__affected_mean_azimuth_abs_offset_deg":
        "Mean absolute horizontal azimuth offset between the UE direction and the serving-cell antenna azimuth during throughput-affected observations (deg).",

    "mob__affected_mean_speed_kmh":
        "Mean UE speed during throughput-affected observations (km/h).",

    "res__mean_rb":
        "Mean number of scheduled resource blocks across all observations.",
}

def format_feature_value(value):
    if pd.isna(value):
        return "missing"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}"
    return str(value)

def build_user_prompt(row, feature_names=SELECTED_FEATURES):
    lines = [
        "Analyze this 5G radio-network incident.",
        "",
        "Diagnostic features:",
    ]
    for feature in feature_names:
        description = FEATURE_DESCRIPTIONS.get(feature, feature)
        lines.append(
            f"- {feature} [{description}]: {format_feature_value(row[feature])}"
        )
    lines += [
        "",
        "Identify the single most likely root cause.",
        "Return only one label from C1 to C8.",
    ]
    return "\n".join(lines)

print(build_user_prompt(train_df.iloc[0]))
print("\nTRUE LABEL:", train_df.iloc[0]["gold_label"])


Analyze this 5G radio-network incident.

Diagnostic features:
- sig__affected_min_rsrp_dbm [Minimum serving-cell RSRP during throughput-affected observations (dBm).]: -88.2100
- tilt__affected_mean_vertical_abs_offset_deg [Mean absolute vertical angular offset between the UE direction and the serving-cell beam direction during throughput-affected observations (deg).]: 12.3861
- sig__mean_rsrp_dbm [Mean serving-cell RSRP across all observations (dBm).]: -79.3550
- tilt__max_serving_total_downtilt_deg [Maximum serving-cell total downtilt, computed as mechanical tilt plus interpreted digital tilt (deg).]: 14.0000
- sig__affected_mean_rsrp_dbm [Mean serving-cell RSRP during throughput-affected observations (dBm).]: -83.2825
- mob__serving_pci_change_rate_per_second [Number of serving PCI changes divided by the incident duration in seconds (changes/s).]: 0.2222
- trn__mean_throughput_delta_when_target_was_prev_neighbor_mbps [Mean throughput change across serving-PCI transitions where the ta

## 5. Qwen2.5-7B-Instruct initialization


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is strongly recommended/required for Qwen2.5-7B fine-tuning. "
        "Run this notebook on a GPU machine."
    )

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        local_files_only=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=compute_dtype,
        device_map="auto",
        trust_remote_code=True,
        local_files_only=True,
    )

print("Model loaded.")

Loading weights: 100%|██████████████████████████████████████████████████████████████| 339/339 [00:02<00:00, 131.16it/s]


Model loaded.


## 6. Prediction and evaluation utilities


In [7]:
def build_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
    ]

@torch.no_grad()
def predict_one(model, row):
    messages = build_messages(row)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt")
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def extract_label(text):
    match = re.search(r"\bC[1-8]\b", str(text).upper())
    return match.group(0) if match else "INVALID"

def evaluate_llm(model, dataframe, max_samples=None, verbose_every=25):
    df = dataframe.copy()
    if max_samples is not None:
        df = df.iloc[:max_samples].copy()

    model.eval()

    rows = []
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        raw_output = predict_one(model, row)
        prediction = extract_label(raw_output)

        rows.append({
            "incident_id": row["incident_id"],
            "true_label": row["gold_label"],
            "prediction": prediction,
            "raw_output": raw_output,
        })

        if verbose_every and i % verbose_every == 0:
            print(f"Processed {i}/{len(df)}")

    results = pd.DataFrame(rows)

    y_true = results["true_label"]
    y_pred = results["prediction"]

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true, y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )

    balanced_acc = recall_score(
        y_true, y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )

    invalid_count = int((results["prediction"] == "INVALID").sum())

    metrics = {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "balanced_accuracy": balanced_acc,
        "invalid_outputs": invalid_count,
        "n_samples": len(results),
    }

    print("\nResults")
    print("-" * 50)
    print(f"Accuracy          : {accuracy:.6f}")
    print(f"Macro-F1          : {macro_f1:.6f}")
    print(f"Balanced Accuracy : {balanced_acc:.6f}")
    print(f"Invalid outputs   : {invalid_count}/{len(results)}")

    print("\nPer-Class Classification Report")
    print("-" * 50)
    print(
        classification_report(
            y_true,
            y_pred,
            labels=LABELS,
            digits=4,
            zero_division=0,
        )
    )

    return results, metrics


## 7. Base-model prediction example

A single validation incident illustrates the model response and label extraction before task-specific fine-tuning.


In [9]:
sample = val_df.iloc[0]

raw_prediction = predict_one(model, sample)

print("Raw Qwen output :", raw_prediction)
print("Parsed label    :", extract_label(raw_prediction))
print("True label      :", sample["gold_label"])


Raw Qwen output : C1
Parsed label    : C1
True label      : C2


## 8. Base-model validation benchmark

The base Qwen2.5-7B-Instruct model is evaluated before fine-tuning to establish a reference for the effect of QLoRA. The saved notebook includes both an initial 50-incident diagnostic run and an evaluation over the entire 339-incident validation set.


In [10]:
BASE_EVAL_MAX_SAMPLES = 50   # Change to None after the sanity run

base_val_results, base_val_metrics = evaluate_llm(
    model,
    val_df,
    max_samples=BASE_EVAL_MAX_SAMPLES,
)

display(pd.DataFrame([base_val_metrics], index=["Base Qwen2.5-7B"]))


Processed 25/50
Processed 50/50

Results
--------------------------------------------------
Accuracy          : 0.060000
Macro-F1          : 0.014151
Balanced Accuracy : 0.125000
Invalid outputs   : 0/50

Per-Class Classification Report
--------------------------------------------------
              precision    recall  f1-score   support

          C1     0.0600    1.0000    0.1132         3
          C2     0.0000    0.0000    0.0000        12
          C3     0.0000    0.0000    0.0000         6
          C4     0.0000    0.0000    0.0000         5
          C5     0.0000    0.0000    0.0000         5
          C6     0.0000    0.0000    0.0000         3
          C7     0.0000    0.0000    0.0000         6
          C8     0.0000    0.0000    0.0000        10

    accuracy                         0.0600        50
   macro avg     0.0075    0.1250    0.0142        50
weighted avg     0.0036    0.0600    0.0068        50



,accuracy,macro_f1,balanced_accuracy,invalid_outputs,n_samples
Base Qwen2.5-7B,0.06,0.014151,0.125,0,50


In [11]:
base_val_results, base_val_metrics = evaluate_llm(
    model,
    val_df,
    max_samples=None,
)

Processed 25/339
Processed 50/339
Processed 75/339
Processed 100/339
Processed 125/339
Processed 150/339
Processed 175/339
Processed 200/339
Processed 225/339
Processed 250/339
Processed 275/339
Processed 300/339
Processed 325/339

Results
--------------------------------------------------
Accuracy          : 0.112094
Macro-F1          : 0.025199
Balanced Accuracy : 0.125000
Invalid outputs   : 0/339

Per-Class Classification Report
--------------------------------------------------
              precision    recall  f1-score   support

          C1     0.1121    1.0000    0.2016        38
          C2     0.0000    0.0000    0.0000        45
          C3     0.0000    0.0000    0.0000        48
          C4     0.0000    0.0000    0.0000        41
          C5     0.0000    0.0000    0.0000        48
          C6     0.0000    0.0000    0.0000        32
          C7     0.0000    0.0000    0.0000        48
          C8     0.0000    0.0000    0.0000        39

    accuracy            

In [12]:
import gc
gc.collect()
torch.cuda.empty_cache()

## 9. Supervised fine-tuning examples

Training examples pair structured incident prompts with their reference root-cause labels.


In [8]:
def build_training_texts(row):
    prompt_messages = build_messages(row)

    full_messages = prompt_messages + [
        {"role": "assistant", "content": str(row["gold_label"]).strip()}
    ]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "prompt_text": prompt_text,
        "full_text": full_text,
    }

train_examples = [
    build_training_texts(row)
    for _, row in train_df.iterrows()
]

val_examples = [
    build_training_texts(row)
    for _, row in val_df.iterrows()
]

train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(train_dataset)
print(val_dataset)


Dataset({
    features: ['prompt_text', 'full_text'],
    num_rows: 1727
})
Dataset({
    features: ['prompt_text', 'full_text'],
    num_rows: 339
})


In [9]:
def tokenize_example(example):
    prompt = tokenizer(
        example["prompt_text"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    full = tokenizer(
        example["full_text"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]
    labels = input_ids.copy()

    prompt_length = min(len(prompt["input_ids"]), len(labels))
    labels[:prompt_length] = [-100] * prompt_length

    # A safety check: at least one answer token must remain supervised.
    if all(x == -100 for x in labels):
        raise ValueError(
            "Prompt consumed the complete sequence. Increase MAX_LENGTH."
        )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

tokenized_val = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
)

print(tokenized_train)
print(tokenized_val)


Map: 100%|███████████████████████████████████████████████████████████████████| 339/339 [00:01<00:00, 302.09 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1727
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 339
})


In [15]:
import gc
gc.collect()
torch.cuda.empty_cache()

## 10. LoRA adapter configuration


In [10]:
if USE_4BIT:
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )

    # PEFT converts non-quantized fp16/bf16 parameters to fp32.
    # Qwen has very large embedding/output matrices, so move these
    # frozen matrices back to the compute dtype to save GPU memory.
    large_frozen_modules = [
        model.get_input_embeddings(),
        model.get_output_embeddings(),
    ]

    for module in large_frozen_modules:
        if module is not None:
            module.to(dtype=compute_dtype)
            for param in module.parameters():
                param.requires_grad_(False)

else:
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("Input embedding dtype :", model.get_input_embeddings().weight.dtype)
print("Output embedding dtype:", model.get_output_embeddings().weight.dtype)

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
Input embedding dtype : torch.bfloat16
Output embedding dtype: torch.bfloat16


In [11]:
import gc
gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()

print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved : {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"Free     : {free / 1024**3:.2f} GB")

Allocated: 5.32 GB
Reserved : 5.34 GB
Free     : 5.92 GB


## 11. Fine-tuning configuration

The reported QLoRA experiment uses:

- **Training duration:** 3 epochs
- **Effective batch size:** 8 (per-device batch size 1, gradient accumulation 8)
- **Learning rate:** `2e-4`
- **Memory optimization:** 4-bit model loading and gradient checkpointing

The validation split is used for development-time evaluation, with the held-out test split reserved for the final assessment.


In [12]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

training_kwargs = dict(
    output_dir="./qwen25_7b_rca_lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=False,
    report_to="none",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=RANDOM_SEED,
)

# Transformers versions use either eval_strategy or evaluation_strategy.
ta_params = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in ta_params:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print(training_args)


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=

In [13]:
import gc
import torch
from transformers import TrainingArguments, Trainer

# Keep a clean copy of the LoRA parameters before the test
initial_trainable_weights = {
    name: param.detach().cpu().clone()
    for name, param in model.named_parameters()
    if param.requires_grad
}

probe_kwargs = training_kwargs.copy()

probe_kwargs["max_steps"] = 1
probe_kwargs["num_train_epochs"] = 1
probe_kwargs["logging_steps"] = 1
probe_kwargs["save_strategy"] = "no"

if "eval_strategy" in probe_kwargs:
    probe_kwargs["eval_strategy"] = "no"

if "evaluation_strategy" in probe_kwargs:
    probe_kwargs["evaluation_strategy"] = "no"

probe_args = TrainingArguments(**probe_kwargs)

probe_trainer = Trainer(
    model=model,
    args=probe_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

model.config.use_cache = False

gc.collect()
torch.cuda.empty_cache()

print("Starting one-step memory test...")

probe_result = probe_trainer.train()

print("\nONE-STEP TRAINING TEST SUCCEEDED")
print(probe_result)

Starting one-step memory test...


Step,Training Loss
1,3.453385



ONE-STEP TRAINING TEST SUCCEEDED
TrainOutput(global_step=1, training_loss=3.4533851146698, metrics={'train_runtime': 10.8202, 'train_samples_per_second': 0.739, 'train_steps_per_second': 0.092, 'total_flos': 351439313777664.0, 'train_loss': 3.4533851146698, 'epoch': 0.004632310364794441})


In [14]:
with torch.no_grad():
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.copy_(
                initial_trainable_weights[name].to(
                    device=param.device,
                    dtype=param.dtype,
                )
            )

model.zero_grad(set_to_none=True)

del probe_trainer
del probe_args
del probe_result
del initial_trainable_weights

gc.collect()
torch.cuda.empty_cache()

print("Probe weights restored.")

free, total = torch.cuda.mem_get_info()
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved : {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"Free     : {free / 1024**3:.2f} GB")

Probe weights restored.
Allocated: 5.33 GB
Reserved : 5.38 GB
Free     : 5.74 GB


## 12. QLoRA fine-tuning

The model is fine-tuned on the eight predefined root-cause classes. The saved training logs and outputs document the completed run.


In [15]:
model.config.use_cache = False

train_result = trainer.train()

print(train_result)


Epoch,Training Loss,Validation Loss
1,0.129072,0.042724
2,0.019599,0.015112
3,0.002222,0.010748


TrainOutput(global_step=648, training_loss=0.07933917562592269, metrics={'train_runtime': 6885.4876, 'train_samples_per_second': 0.752, 'train_steps_per_second': 0.094, 'total_flos': 2.2743495242741146e+17, 'train_loss': 0.07933917562592269, 'epoch': 3.0})


## 13. Trained adapter checkpoint


In [16]:
LORA_OUTPUT_DIR = Path("./qwen25_7b_rca_lora_final")

trainer.model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print("Saved LoRA adapter/tokenizer to:", LORA_OUTPUT_DIR.resolve())


Saved LoRA adapter/tokenizer to: /home/azaddel/5GLab/llm/qwen25_7b_rca_lora_final


In [20]:
import gc
import torch

model.config.use_cache = True
model.eval()

gc.collect()
torch.cuda.empty_cache()

print("Input embedding dtype :", model.get_input_embeddings().weight.dtype)
print("Output embedding dtype:", model.get_output_embeddings().weight.dtype)
print("Compute dtype         :", compute_dtype)

Input embedding dtype : torch.bfloat16
Output embedding dtype: torch.bfloat16
Compute dtype         : torch.bfloat16


In [21]:
@torch.no_grad()
def predict_one(model, row):
    messages = build_messages(row)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt")
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.autocast(
        device_type="cuda",
        dtype=compute_dtype,
    ):
        output_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

In [22]:
print("Testing one BASE-model prediction...")

with model.disable_adapter():
    test_output = predict_one(model, val_df.iloc[0])

print("Output:", test_output)
print("Parsed:", extract_label(test_output))
print("True  :", val_df.iloc[0]["gold_label"])

Testing one BASE-model prediction...
Output: C1
Parsed: C1
True  : C2


In [23]:
import gc
gc.collect()
torch.cuda.empty_cache()

print("Evaluating BASE model on full validation set...")

with model.disable_adapter():
    base_val_results, base_val_metrics = evaluate_llm(
        model,
        val_df,
        max_samples=None,
    )

print("\nBase-model metrics:")
print(base_val_metrics)

Evaluating BASE model on full validation set...
Processed 25/339
Processed 50/339
Processed 75/339
Processed 100/339
Processed 125/339
Processed 150/339
Processed 175/339
Processed 200/339
Processed 225/339
Processed 250/339
Processed 275/339
Processed 300/339
Processed 325/339

Results
--------------------------------------------------
Accuracy          : 0.112094
Macro-F1          : 0.025199
Balanced Accuracy : 0.125000
Invalid outputs   : 0/339

Per-Class Classification Report
--------------------------------------------------
              precision    recall  f1-score   support

          C1     0.1121    1.0000    0.2016        38
          C2     0.0000    0.0000    0.0000        45
          C3     0.0000    0.0000    0.0000        48
          C4     0.0000    0.0000    0.0000        41
          C5     0.0000    0.0000    0.0000        48
          C6     0.0000    0.0000    0.0000        32
          C7     0.0000    0.0000    0.0000        48
          C8     0.0000    0.00

In [24]:
import gc
gc.collect()
torch.cuda.empty_cache()

print("LoRA adapter should now be active again.")

LoRA adapter should now be active again.


## 14. Fine-tuned model: validation performance

The tuned model is evaluated on the full validation split and compared against the base-model reference.


In [25]:
model.config.use_cache = True
model.eval()

lora_val_results, lora_val_metrics = evaluate_llm(
    model,
    val_df,
    max_samples=None,
)

comparison = pd.DataFrame(
    [base_val_metrics, lora_val_metrics],
    index=["Base Qwen2.5-7B", "Qwen2.5-7B + LoRA"],
)

display(comparison)


Processed 25/339
Processed 50/339
Processed 75/339
Processed 100/339
Processed 125/339
Processed 150/339
Processed 175/339
Processed 200/339
Processed 225/339
Processed 250/339
Processed 275/339
Processed 300/339
Processed 325/339

Results
--------------------------------------------------
Accuracy          : 0.988201
Macro-F1          : 0.987950
Balanced Accuracy : 0.987422
Invalid outputs   : 0/339

Per-Class Classification Report
--------------------------------------------------
              precision    recall  f1-score   support

          C1     0.9737    0.9737    0.9737        38
          C2     1.0000    0.9778    0.9888        45
          C3     0.9592    0.9792    0.9691        48
          C4     0.9762    1.0000    0.9880        41
          C5     1.0000    1.0000    1.0000        48
          C6     1.0000    0.9688    0.9841        32
          C7     1.0000    1.0000    1.0000        48
          C8     1.0000    1.0000    1.0000        39

    accuracy            

,accuracy,macro_f1,balanced_accuracy,invalid_outputs,n_samples
Base Qwen2.5-7B,0.112094,0.025199,0.125000,0,339
Qwen2.5-7B + LoRA,0.988201,0.987950,0.987422,0,339


In [26]:
import gc
gc.collect()
torch.cuda.empty_cache()

## 15. Final held-out test evaluation

The held-out test split provides the final assessment of eight-class root-cause classification performance.


In [27]:
lora_test_results, lora_test_metrics = evaluate_llm(
    model,
    test_df,
    max_samples=None,
)

display(pd.DataFrame([lora_test_metrics], index=["Qwen2.5-7B + LoRA — Test"]))


Processed 25/334
Processed 50/334
Processed 75/334
Processed 100/334
Processed 125/334
Processed 150/334
Processed 175/334
Processed 200/334
Processed 225/334
Processed 250/334
Processed 275/334
Processed 300/334
Processed 325/334

Results
--------------------------------------------------
Accuracy          : 0.988024
Macro-F1          : 0.987671
Balanced Accuracy : 0.986677
Invalid outputs   : 0/334

Per-Class Classification Report
--------------------------------------------------
              precision    recall  f1-score   support

          C1     0.9722    0.9459    0.9589        37
          C2     1.0000    1.0000    1.0000        45
          C3     0.9388    0.9787    0.9583        47
          C4     1.0000    1.0000    1.0000        41
          C5     1.0000    1.0000    1.0000        46
          C6     1.0000    0.9688    0.9841        32
          C7     1.0000    1.0000    1.0000        46
          C8     1.0000    1.0000    1.0000        40

    accuracy            

,accuracy,macro_f1,balanced_accuracy,invalid_outputs,n_samples
Qwen2.5-7B + LoRA — Test,0.988024,0.987671,0.986677,0,334


## 16. Exported predictions and metrics


In [28]:
lora_val_results.to_csv(
    "qwen25_7b_lora_validation_predictions.csv",
    index=False,
)

lora_test_results.to_csv(
    "qwen25_7b_lora_test_predictions.csv",
    index=False,
)

metrics_table = pd.DataFrame(
    [
        {"model": "Base Qwen2.5-7B", "split": "validation", **base_val_metrics},
        {"model": "Qwen2.5-7B + LoRA", "split": "validation", **lora_val_metrics},
        {"model": "Qwen2.5-7B + LoRA", "split": "test", **lora_test_metrics},
    ]
)

metrics_table.to_csv(
    "qwen25_7b_lora_metrics.csv",
    index=False,
)

display(metrics_table)


,model,split,accuracy,macro_f1,balanced_accuracy,invalid_outputs,n_samples
0,Base Qwen2.5-7B,validation,0.112094,0.025199,0.125000,0,339
1,Qwen2.5-7B + LoRA,validation,0.988201,0.987950,0.987422,0,339
2,Qwen2.5-7B + LoRA,test,0.988024,0.987671,0.986677,0,334


In [30]:
print(lora_test_results.columns.tolist())

['incident_id', 'true_label', 'prediction', 'raw_output']


In [31]:
test_errors = lora_test_results[
    lora_test_results["prediction"] != lora_test_results["true_label"]
].copy()

print(f"Number of test errors: {len(test_errors)}")
display(test_errors)

Number of test errors: 4


,incident_id,true_label,prediction,raw_output
68,ID_L7MAVBLH9Q,C3,C1,C1
205,ID_ERUCREN2XX,C6,C3,C3
239,ID_WX66QMBSBX,C1,C3,C3
323,ID_BY391F7N9F,C1,C3,C3


## Results and methodological considerations

On the **339-incident validation set**, accuracy increased from **11.21%** for the base Qwen2.5-7B model to **98.82%** after QLoRA fine-tuning. The fine-tuned model achieved **98.80%** accuracy on the **334-incident held-out test set**. These figures describe performance on the specific dataset and predefined eight-class task evaluated here.

An important dataset caveat is the unusually strong class-specific structure previously observed in `mob__serving_pci_change_rate_per_second`, particularly for C2 and C5. A separate feature-ablation study would help determine how much performance depends on that signal. Such an ablation is **not part of the completed results shown here**, and test-set outcomes should not be used to tune the reported pipeline.
